# 🎾 OMNIS-COURT LLM + Jina Server (v7.3 STABLE)
## Optimized for Colab T4 Free Tier

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run All (Ctrl+F9)
3. Wait ~5-8 minutes
4. Copy both URLs from Cell 5
5. Paste into config/platforms.json
6. Close tab (anti-idle active)

In [ ]:
# ==========================================
# CELL 1: ENVIRONMENT + INSTALL
# ==========================================
import os

# Force CUDA platform BEFORE any import
os.environ['VLLM_TARGET_DEVICE'] = 'cuda'
os.environ['VLLM_PLATFORM'] = 'cuda'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Clean any conflicting packages
!pip uninstall -y vllm flashinfer-python humming-kernels 2>/dev/null

# Install vLLM 0.8.4 (stable, supports Qwen3, CUDA 12.x compatible)
# This version is tested with Colab T4 Free Tier
!pip install "vllm==0.8.4" --no-cache-dir 2>&1 | tail -5

# Install supporting packages
!pip install -q trafilatura fastapi uvicorn nest-asyncio requests

# Install cloudflared binary (NOT pip package)
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Verify ALL installations
import subprocess
cf = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f'\n✅ cloudflared: {cf.stdout.strip()}')

all_ok = True
for pkg in ['vllm', 'trafilatura', 'fastapi', 'uvicorn']:
    try:
        __import__(pkg.replace('-','_'))
        print(f'✅ {pkg}')
    except Exception as e:
        print(f'❌ {pkg}: {e}')
        all_ok = False

if all_ok:
    print('\n✅ ALL dependencies ready!')
    print('⚠️  NOW: Runtime → Restart runtime → Then Run All again')
else:
    print('\n❌ SOME packages failed. STOP here and send error.')

In [ ]:
# ==========================================
# CELL 2: RE-SET ENV + ANTI-IDLE
# ==========================================
import os
# Must re-set env vars after restart
os.environ['VLLM_TARGET_DEVICE'] = 'cuda'
os.environ['VLLM_PLATFORM'] = 'cuda'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print('✅ Environment variables set (CUDA forced)')

from IPython.display import display, Javascript

# Anti-idle: auto-click run button every 5 minutes
display(Javascript('''
    setInterval(function(){
        var btn = document.querySelector('colab-run-button');
        if(btn) btn.click();
        console.log('[Anti-Idle] Ping at ' + new Date().toLocaleTimeString());
    }, 300000);
'''))
print('✅ Anti-idle active! Safe to close tab after all cells run.')

In [ ]:
# ==========================================
# CELL 3: START QWEN3-30B-A3B VLLM SERVER
# ==========================================
import subprocess, time, requests

# Qwen3-30B-A3B is MoE: 30B total params but only 3B active per token
# Fits in T4 16GB VRAM with max-model-len 8192
proc = subprocess.Popen([
    'python','-m','vllm.entrypoints.openai.api_server',
    '--model','Qwen/Qwen3-30B-A3B',
    '--served-model-name','qwen3-30b',
    '--host','0.0.0.0','--port','8000',
    '--max-model-len','8192',
    '--gpu-memory-utilization','0.9',
    '--trust-remote-code',
    '--enforce-eager',
    '--dtype','bfloat16'
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print('🚀 Starting vLLM with Qwen3-30B-A3B... (~5-8 min)')
for i in range(60):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'✅ vLLM READY on port 8000 ({(i+1)*10}s)')
            break
    except:
        pass
    time.sleep(10)
    if i % 6 == 0:
        print(f'⏳ Waiting... {(i+1)*10}s')
else:
    print('❌ Failed after 600s. Last logs:')
    print(proc.stderr.read().decode()[-1500:])

In [ ]:
# ==========================================
# CELL 4: START JINA READER SERVER
# ==========================================
import threading, time, requests as req
import trafilatura
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn, nest_asyncio
nest_asyncio.apply()

app = FastAPI(title='OMNIS Jina Reader')

@app.get('/health')
async def health():
    return {'status':'ok'}

@app.get('/extract')
async def extract(url: str = Query(...)):
    try:
        dl = trafilatura.fetch_url(url)
        if not dl:
            return JSONResponse(400, content={'error':'fetch failed','url':url})
        txt = trafilatura.extract(dl, include_comments=False, include_tables=True, no_fallback=False)
        if not txt or len(txt.strip()) < 50:
            return JSONResponse(400, content={'error':'content too short','url':url})
        return {'url':url,'content':txt,'word_count':len(txt.split()),'status':'success'}
    except Exception as e:
        return JSONResponse(500, content={'error':str(e),'url':url})

def run():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='warning')

t = threading.Thread(target=run, daemon=True)
t.start()
time.sleep(3)
try:
    r = req.get('http://localhost:8001/health', timeout=5)
    print('✅ Jina Reader READY on port 8001' if r.status_code==200 else '❌ Jina error')
except Exception as e:
    print(f'❌ Jina failed: {e}')

In [ ]:
# ==========================================
# CELL 5: CLOUDFLARE TUNNELS (2 URLs)
# ==========================================
import subprocess, re

def tunnel(port):
    p = subprocess.Popen(
        ['cloudflared','tunnel','--url',f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in p.stderr:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            return p, m.group(0)
    return p, None

print('🌐 Tunnel LLM (8000)...')
p1, u1 = tunnel(8000)
print('🌐 Tunnel Jina (8001)...')
p2, u2 = tunnel(8001)

if u1 and u2:
    print('\n' + '='*60)
    print('🎉 OMNIS-COURT COLAB READY!')
    print('='*60)
    print(f'🧠 LLM:  {u1}')
    print(f'📖 JINA: {u2}')
    print('='*60)
    print('📋 COPY BOTH URLs → config/platforms.json')
    print('🔒 Anti-idle ON → safe to close tab')
    print('🧪 Test URLs from YOUR browser (not from Colab)')
    print('='*60)
else:
    print(f'❌ Tunnel failed: LLM={u1}, Jina={u2}')

In [ ]:
# ==========================================
# CELL 6: LOCALHOST TESTS ONLY
# ==========================================
# Note: Tunnel URLs cannot be tested from within Colab itself
# due to Cloudflare DNS restrictions. Test from your browser instead.

import requests

print('🧪 Testing vLLM on localhost:8000...')
try:
    r = requests.post(
        'http://localhost:8000/v1/chat/completions',
        json={'model':'qwen3-30b','messages':[{'role':'user','content':'Say OK'}],'max_tokens':5},
        timeout=30
    )
    if r.status_code == 200:
        print(f"✅ LLM localhost OK: {r.json()['choices'][0]['message']['content']}")
    else:
        print(f'❌ LLM localhost: {r.status_code}')
except Exception as e:
    print(f'❌ LLM localhost: {e}')

print('\n🧪 Testing Jina on localhost:8001...')
try:
    r = requests.get(
        'http://localhost:8001/extract',
        params={'url':'https://en.wikipedia.org/wiki/Tennis'},
        timeout=30
    )
    if r.status_code == 200:
        print(f"✅ Jina localhost OK: {r.json()['word_count']} words")
    else:
        print(f'❌ Jina localhost: {r.status_code}')
except Exception as e:
    print(f'❌ Jina localhost: {e}')

print('\n' + '='*60)
print('🌐 NOW TEST TUNNEL URLs FROM YOUR BROWSER:')
print(f'   {u1}/v1/models')
print(f'   {u2}/health')
print('='*60)